<a href="https://colab.research.google.com/github/EliVil2/RUTAS-IO/blob/main/Intento_2_IO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install openrouteservice folium geopy ortools

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 81.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.6/135.6 kB 10.4 MB/s eta 0:00:00
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0


In [5]:
# === IMPORTACIONES ===
import folium
import openrouteservice
from openrouteservice import convert
import numpy as np
from collections import deque

# === CONFIGURACIÓN ===
ORS_API_KEY = '5b3ce3597851110001cf6248b75b48749e4e41a38cdaea0746c5ae37'
client = openrouteservice.Client(key=ORS_API_KEY)

# === DATOS DE ENTRADA ===
locations = [
    (10.983724, -74.789957),  # Origen - Universidad sede centro
    (10.9700532, -74.8019507), (10.9525577, -74.8039492), (10.975318, -74.808005),
    (10.9066255, -74.7856385), (10.9724651, -74.8060456), (11.017676, -74.809716),
    (10.944379, -74.803011), (10.9124556, -74.7826908), (10.9606144, -74.8028657),
    (10.9230761, -74.8008353), (10.9628562, -74.8349773), (10.940902, -74.771705),
    (10.9942585, -74.8122456), (10.963991, -74.817237), (11.027218, -74.869495),
    (10.9615811, -74.8302943), (10.979663, -74.80365), (10.9992743, -74.7924869),
    (10.9625489, -74.830824), (10.985877, -74.83556), (11.024694, -74.869555),
    (10.932394, -74.766357), (11.023623, -74.80664), (10.9030214, -74.7919232),
    (11.02425, -74.86800), (10.9558219, -74.8218484), (10.9599135, -74.8071882),
    (10.894661, -74.885339), (10.9590029, -74.796548), (10.7465427, -74.7565432),
    (11.0218788, -74.8705983),
]
demands = [0, 107, 112, 95, 84, 72, 107, 52, 110, 49, 67, 75, 107, 111, 107, 80, 87, 94, 78, 66, 30,
           70, 69, 94, 70, 114, 64, 46, 76, 33, 60, 69]
vehicle_capacity = 500

# === FUNCIONES ===
def get_route(coords):
    try:
        route = client.directions(coords, profile='driving-car', format='geojson')
        distance = route['features'][0]['properties']['segments'][0]['distance']
        geometry = route['features'][0]['geometry']
        return distance, geometry
    except Exception as e:
        print("Error con ruta:", coords)
        return float('inf'), None

def nearest_neighbor_route(cluster, locations):
    unvisited = set(cluster)
    current = 0  # Inicio en depósito
    route = [0]
    while unvisited:
        next_stop = min(unvisited, key=lambda x: np.linalg.norm(np.array(locations[current]) - np.array(locations[x])))
        route.append(next_stop)
        unvisited.remove(next_stop)
        current = next_stop
    return route

# === AGRUPACIÓN GREEDY CON RESPETO A CAPACIDAD ===
remaining = deque(sorted(range(1, len(locations)), key=lambda x: -demands[x]))
clusters = []
current_cluster = []
current_weight = 0

while remaining:
    idx = remaining.popleft()
    if current_weight + demands[idx] <= vehicle_capacity:
        current_cluster.append(idx)
        current_weight += demands[idx]
    else:
        clusters.append(current_cluster)
        current_cluster = [idx]
        current_weight = demands[idx]
if current_cluster:
    clusters.append(current_cluster)

print(f"Total vehículos asignados: {len(clusters)}")

# === GENERAR RUTAS Y MAPA ===
m = folium.Map(location=locations[0], zoom_start=12)
colors = ['blue', 'green', 'red', 'purple', 'orange', 'darkred', 'cadetblue', 'pink', 'darkgreen']
all_routes = []
total_distance = 0

for i, cluster in enumerate(clusters):
    route = nearest_neighbor_route(cluster, locations)
    route_coords = [locations[route[j]] for j in range(len(route))]
    total_weight = sum(demands[j] for j in route if j != 0)
    distance = 0

    for j in range(len(route_coords) - 1):
        segment = [tuple(reversed(route_coords[j])), tuple(reversed(route_coords[j + 1]))]
        dist, geometry = get_route(segment)
        distance += dist
        if geometry:
            folium.GeoJson(geometry, name=f"Ruta Vehículo {i + 1}",
                           style_function=lambda x, color=colors[i % len(colors)]: {'color': color, 'weight': 4}).add_to(m)

    total_distance += distance
    all_routes.append((i + 1, route, total_weight, distance / 1000))

# === MOSTRAR RESULTADOS ===
for veh_id, route, weight, dist_km in all_routes:
    print(f"Vehículo {veh_id}: Ruta: {route} | Peso total: {weight} kg | Distancia: {dist_km:.2f} km")
    if weight > vehicle_capacity:
        print(f"⚠ ADVERTENCIA: Ruta del vehículo {veh_id} excede la capacidad!")

m.save("rutas_vehiculos.html")
print("Mapa guardado como rutas_vehiculos.html")


Total vehículos asignados: 6


/usr/local/lib/python3.11/dist-packages/openrouteservice/client.py:211: UserWarning: Rate limit exceeded. Retrying for the 1st time.
  warnings.warn('Rate limit exceeded. Retrying for the {0}{1} time.'.format(retry_counter + 1,
/usr/local/lib/python3.11/dist-packages/openrouteservice/client.py:211: UserWarning: Rate limit exceeded. Retrying for the 2nd time.
  warnings.warn('Rate limit exceeded. Retrying for the {0}{1} time.'.format(retry_counter + 1,
/usr/local/lib/python3.11/dist-packages/openrouteservice/client.py:211: UserWarning: Rate limit exceeded. Retrying for the 3rd time.
  warnings.warn('Rate limit exceeded. Retrying for the {0}{1} time.'.format(retry_counter + 1,
/usr/local/lib/python3.11/dist-packages/openrouteservice/client.py:211: UserWarning: Rate limit exceeded. Retrying for the 4th time.
  warnings.warn('Rate limit exceeded. Retrying for the {0}{1} time.'.format(retry_counter + 1,
/usr/local/lib/python3.11/dist-packages/openrouteservice/client.py:211: UserWarning: Rat

Vehículo 1: Ruta: [0, 13, 2, 8, 25] | Peso total: 447 kg | Distancia: 39.83 km
Vehículo 2: Ruta: [0, 1, 14, 12, 6] | Peso total: 428 kg | Distancia: 25.19 km
Vehículo 3: Ruta: [0, 17, 3, 16, 23, 4] | Peso total: 454 kg | Distancia: 33.14 km
Vehículo 4: Ruta: [0, 18, 5, 11, 21, 15, 28] | Peso total: 451 kg | Distancia: 41.23 km
Vehículo 5: Ruta: [0, 26, 19, 10, 24, 22, 31, 30] | Peso total: 465 kg | Distancia: 91.43 km
Vehículo 6: Ruta: [0, 29, 9, 27, 7, 20] | Peso total: 210 kg | Distancia: 19.46 km
Mapa guardado como rutas_vehiculos.html
